# 🤖 AI Stock Predictor - Interactive Notebook

This notebook provides an interactive interface similar to the Streamlit GUI for the AI Stock Predictor system.

## Features:
- 📊 Data Collection (Price & News)
- 🔑 API Key Status Check
- 📈 Data Visualization
- 📰 News Analysis
- 🤖 Model Training
- 🔮 Price Predictions

---

## 🚀 Google Colab Setup

**If you're using Google Colab, follow these steps first:**

1. **Upload `ai_stock_predictor.py`**:
   - Run the upload cell below (Section 0.5)
   - Or use: Files → Upload to session storage

2. **Set up API Keys** (Section 3):
   - Use Colab Secrets (🔑 icon in sidebar) - Recommended
   - Or set environment variables directly

3. **Run all cells in order**

---

## 0.5. Upload Files (Google Colab Only)

**Skip this section if running locally. For Colab, upload `ai_stock_predictor.py` here:**

In [ ]:
# Upload ai_stock_predictor.py (Google Colab only)
# Uncomment and run this cell if using Google Colab
from google.colab import files
uploaded = files.upload()  # Select ai_stock_predictor.py

# For local Jupyter, make sure ai_stock_predictor.py is in the same directory

In [ ]:
## 1. Installation & Setup

**Run this cell first to install packages and import libraries:**

# Install required packages (run this first if needed)
# Uncomment for Google Colab or if packages are missing
# !pip install torch pandas numpy yfinance requests plotly ipywidgets python-dotenv transformers

In [ ]:
# Import libraries
import os  # Important: os must be imported first
import sys
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For Google Colab: Add current directory to path if needed
# import sys
# sys.path.append('/content')  # Uncomment if import fails in Colab

# Import the AI Stock Predictor
try:
    from ai_stock_predictor import (
        AIStockPredictor,
        TimeFrame,
        MarketRegime,
        Prediction
    )
    print("✅ Libraries imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Make sure ai_stock_predictor.py is in the same directory")
    print("   In Colab: Upload the file using the upload cell above")

## 2. Configuration

In [ ]:
# Configuration
SYMBOL = "BTC-USD"  # Change this to your desired symbol (e.g., "ETH-USD", "AAPL", "TSLA")
PERIOD = "6mo"  # Options: "1mo", "3mo", "6mo", "1y", "2y"
INTERVAL = "1d"  # Options: "1h", "1d", "1wk"
NEWS_DAYS = 30  # Number of days to look back for news
MAX_ARTICLES_PER_SOURCE = 1000  # Maximum articles per source

print(f"📊 Configuration:")
print(f"  Symbol: {SYMBOL}")
print(f"  Period: {PERIOD}")
print(f"  Interval: {INTERVAL}")
print(f"  News Lookback: {NEWS_DAYS} days")
print(f"  Max Articles per Source: {MAX_ARTICLES_PER_SOURCE}")

In [ ]:
# Load .env file if it exists (for local use)
import os  # Make sure os is imported

try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✅ .env file loaded")
except ImportError:
    print("⚠️ python-dotenv not installed. Install with: pip install python-dotenv")
except Exception as e:
    print(f"⚠️ Could not load .env file: {e}")

# For Google Colab: Load API keys from Colab Secrets
try:
    from google.colab import userdata
    print("\n🔑 Loading API keys from Colab Secrets...")
    for key_name in ['NEWSAPI_KEY', 'ALPHAVANTAGE_API_KEY', 'FINNHUB_API_KEY',
                     'POLYGON_API_KEY', 'TWITTER_BEARER_TOKEN', 'REDDIT_CLIENT_ID',
                     'NEWSCATCHER_API_KEY', 'BING_SEARCH_API_KEY']:
        try:
            key_value = userdata.get(key_name)
            if key_value:
                os.environ[key_name] = str(key_value).strip()
                print(f"  ✅ {key_name} loaded")
        except:
            pass
    print("💡 To add keys: Click 🔑 icon (Secrets) in Colab sidebar")
except ImportError:
    print("💡 Not in Colab - using .env file or environment variables")
except Exception as e:
    print(f"⚠️ Could not load Colab secrets: {e}")

## 4. Initialize Predictor

In [ ]:
# Check API Key Status
import os  # Make sure os is imported

api_keys_status = {
    'NewsAPI': ('NEWSAPI_KEY', 'https://newsapi.org/register', '100/day free'),
    'Alpha Vantage': ('ALPHAVANTAGE_API_KEY', 'https://www.alphavantage.co/support/#api-key', '500/day free'),
    'Finnhub': ('FINNHUB_API_KEY', 'https://finnhub.io/register', '60/min free'),
    'Polygon.io': ('POLYGON_API_KEY', 'https://polygon.io/', 'Paid'),
    'Twitter/X': ('TWITTER_BEARER_TOKEN', 'https://developer.twitter.com/', 'Limited free'),
    'Reddit': ('REDDIT_CLIENT_ID', 'https://www.reddit.com/prefs/apps', 'Free'),
    'NewsCatcher': ('NEWSCATCHER_API_KEY', 'https://newscatcher.ai/', '100/month free'),
    'Bing News': ('BING_SEARCH_API_KEY', 'https://www.microsoft.com/en-us/bing/apis/bing-news-search-api', 'Azure account'),
}

keys_found = []
keys_missing = []

for service_name, (key_name, url, tier) in api_keys_status.items():
    key_value = os.getenv(key_name, '')
    if key_value:
        masked_key = key_value[:4] + "..." + key_value[-4:] if len(key_value) > 8 else "***"
        keys_found.append((service_name, masked_key, tier))
    else:
        keys_missing.append((service_name, url, tier))

print("🔑 API Key Status:")
print("=" * 60)

if keys_found:
    print(f"\n✅ {len(keys_found)} API key(s) loaded:")
    for service_name, masked_key, tier in keys_found:
        print(f"  • {service_name} ({tier}) - Key: {masked_key}")

if keys_missing:
    print(f"\n⚠️ {len(keys_missing)} API key(s) missing:")
    for service_name, url, tier in keys_missing[:5]:
        print(f"  • {service_name} - Get key: {url} ({tier})")
    if len(keys_missing) > 5:
        print(f"  ... and {len(keys_missing) - 5} more")

print("\n💡 Free sources (no API key needed):")
print("  • yfinance (Yahoo Finance)")
print("  • CryptoCompare (for crypto)")

if keys_found:
    print("\n🎉 You have API keys! You can collect 200-4000+ articles.")
else:
    print("\n⚠️ No API keys found. You'll only get ~50 articles from free sources.")
    print("   For Colab: Use Secrets (🔑 icon) or add to .env file")

## 4. Initialize Predictor

In [ ]:
# Initialize the AI Stock Predictor
predictor = AIStockPredictor(SYMBOL)
print(f"✅ Predictor initialized for {SYMBOL}")

## 5. Collect Data

In [ ]:
# Collect all data (price, news, technical indicators, etc.)
print("📊 Collecting data...")
print("This may take a few minutes depending on your internet connection and API limits.")

features = predictor.collect_data(
    period=PERIOD,
    interval=INTERVAL,
    news_days=NEWS_DAYS,
    max_news_per_source=MAX_ARTICLES_PER_SOURCE
)

print("\n✅ Data collection complete!")
print(f"  Price data points: {len(predictor.price_data)}")
print(f"  News articles: {len(predictor.news_agent.news_cache)}")
print(f"  Features: {len(features.columns)}")

## 6. Data Summary & Statistics

In [ ]:
# Display collection statistics
stats = predictor.news_agent.collection_stats

print("📊 Collection Statistics:")
print("=" * 60)
print(f"Sources used: {stats.get('sources_used', [])}")
print(f"Sources failed: {stats.get('sources_failed', [])}")
print(f"Total collected (before dedup): {stats.get('total_collected', 0)}")
print(f"Duplicates removed: {stats.get('duplicates_removed', 0)}")
print(f"Final unique articles: {len(predictor.news_agent.news_cache)}")

# Show articles by source
if predictor.news_agent.news_cache:
    from collections import Counter
    sources = [item.get('source', 'Unknown') for item in predictor.news_agent.news_cache]
    source_counts = Counter(sources)

    print("\n📰 Articles by source:")
    for source, count in source_counts.most_common(10):
        print(f"  • {source}: {count} articles")

In [ ]:
# Plot price chart
fig = go.Figure()

# Candlestick chart
fig.add_trace(go.Candlestick(
    x=price_data['datetime'],
    open=price_data['open'],
    high=price_data['high'],
    low=price_data['low'],
    close=price_data['close'],
    name='Price'
))

fig.update_layout(
    title=f'{SYMBOL} Price Chart',
    xaxis_title='Date',
    yaxis_title='Price',
    height=500,
    template='plotly_white'
)

fig.show()

## 8. News Data Analysis

In [ ]:
# Plot price chart
fig = go.Figure()

# Candlestick chart
fig.add_trace(go.Candlestick(
    x=price_data['datetime'],
    open=price_data['open'],
    high=price_data['high'],
    low=price_data['low'],
    close=price_data['close'],
    name='Price'
))

fig.update_layout(
    title=f'{SYMBOL} Price Chart',
    xaxis_title='Date',
    yaxis_title='Price',
    height=500,
    template='plotly_white'
)

fig.show()

In [ ]:
# News statistics by source
if news_data:
    from collections import Counter
    sources = [item.get('source', 'Unknown') for item in news_data]
    source_counts = Counter(sources)

    # Bar chart of articles by source
    fig = go.Figure(data=[
        go.Bar(
            x=list(source_counts.keys())[:15],
            y=list(source_counts.values())[:15],
            marker_color='lightblue'
        )
    ])

    fig.update_layout(
        title='News Articles by Source',
        xaxis_title='Source',
        yaxis_title='Number of Articles',
        height=400,
        template='plotly_white',
        xaxis_tickangle=-45
    )

    fig.show()

In [ ]:
# Convert news data to DataFrame
news_data = predictor.news_agent.news_cache
if news_data:
    news_df = pd.DataFrame(news_data)
    print(f"News Data Shape: {news_df.shape}")
    print("\nColumns:", news_df.columns.tolist())
    print("\nFirst few articles:")
    display(news_df[['timestamp', 'title', 'source', 'category', 'sentiment']].head(10))
else:
    print("No news data available")

## 9. Feature Analysis

In [ ]:
# Display features
features = predictor.features
if features is not None:
    print(f"Features Shape: {features.shape}")
    print(f"\nFeature Categories:")

    # Categorize features
    technical_features = [col for col in features.columns if any(x in col.lower() for x in ['rsi', 'macd', 'bollinger', 'ema', 'sma', 'atr', 'stoch'])]
    time_features = [col for col in features.columns if any(x in col.lower() for x in ['hour', 'day', 'month', 'week', 'lag'])]
    news_features = [col for col in features.columns if any(x in col.lower() for x in ['news', 'sentiment'])]
    macro_features = [col for col in features.columns if any(x in col.lower() for x in ['fear', 'greed', 'dxy', 'macro'])]
    other_features = [col for col in features.columns if col not in technical_features + time_features + news_features + macro_features and col not in ['datetime', 'open', 'high', 'low', 'close', 'volume']]

    print(f"  Technical Indicators: {len(technical_features)}")
    print(f"  Time & Lag Features: {len(time_features)}")
    print(f"  News & Sentiment: {len(news_features)}")
    print(f"  Macro Features: {len(macro_features)}")
    print(f"  Other Features: {len(other_features)}")

    print("\nSample features:")
    display(features.head())
else:
    print("No features available")

## 10. Export Data

In [ ]:
# Display features
features = predictor.features
if features is not None:
    print(f"Features Shape: {features.shape}")
    print(f"\nFeature Categories:")

    # Categorize features
    technical_features = [col for col in features.columns if any(x in col.lower() for x in ['rsi', 'macd', 'bollinger', 'ema', 'sma', 'atr', 'stoch'])]
    time_features = [col for col in features.columns if any(x in col.lower() for x in ['hour', 'day', 'month', 'week', 'lag'])]
    news_features = [col for col in features.columns if any(x in col.lower() for x in ['news', 'sentiment'])]
    macro_features = [col for col in features.columns if any(x in col.lower() for x in ['fear', 'greed', 'dxy', 'macro'])]
    other_features = [col for col in features.columns if col not in technical_features + time_features + news_features + macro_features and col not in ['datetime', 'open', 'high', 'low', 'close', 'volume']]

    print(f"  Technical Indicators: {len(technical_features)}")
    print(f"  Time & Lag Features: {len(time_features)}")
    print(f"  News & Sentiment: {len(news_features)}")
    print(f"  Macro Features: {len(macro_features)}")
    print(f"  Other Features: {len(other_features)}")

    print("\nSample features:")
    display(features.head())
else:
    print("No features available")

## 10. Export Data

In [ ]:
# Export data to CSV
export_dir = "exports"
os.makedirs(export_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export price data
if predictor.price_data is not None:
    price_file = f"{export_dir}/price_data_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    predictor.price_data.to_csv(price_file, index=False)
    print(f"✅ Price data exported to: {price_file}")

# Export news data
if predictor.news_agent.news_cache:
    news_df = pd.DataFrame(predictor.news_agent.news_cache)
    news_file = f"{export_dir}/news_data_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    news_df.to_csv(news_file, index=False)
    print(f"✅ News data exported to: {news_file}")

# Export features
if predictor.features is not None:
    features_file = f"{export_dir}/features_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    predictor.features.to_csv(features_file, index=False)
    print(f"✅ Features exported to: {features_file}")

# For Google Colab: Download files
try:
    from google.colab import files
    print("\n💡 To download files in Colab, use:")
    print(f"   files.download('{price_file}')")
    print(f"   files.download('{news_file}')")
    print(f"   files.download('{features_file}')")
except ImportError:
    print("\n💡 Files saved locally. In Colab, use files.download() to download.")

---

## 📝 Notes

- **API Keys**:
  - **Colab**: Use Secrets (🔑 icon) - Recommended
  - **Local**: Add to `.env` file
- **Free Sources**: yfinance and CryptoCompare work without API keys
- **Training**: Model training is optional and can take a long time
- **Predictions**: Requires trained models

For more information, see:
- `GOOGLE_COLAB_SETUP.md` - Detailed Colab setup guide
- `ai_stock_predictor_gui.py` - Streamlit GUI version

---

In [ ]:
# Export data to CSV
export_dir = "exports"
os.makedirs(export_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export price data
if predictor.price_data is not None:
    price_file = f"{export_dir}/price_data_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    predictor.price_data.to_csv(price_file, index=False)
    print(f"✅ Price data exported to: {price_file}")

# Export news data
if predictor.news_agent.news_cache:
    news_df = pd.DataFrame(predictor.news_agent.news_cache)
    news_file = f"{export_dir}/news_data_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    news_df.to_csv(news_file, index=False)
    print(f"✅ News data exported to: {news_file}")

# Export features
if predictor.features is not None:
    features_file = f"{export_dir}/features_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    predictor.features.to_csv(features_file, index=False)
    print(f"✅ Features exported to: {features_file}")

---

## 📝 Notes

- **API Keys**: Add your API keys to a `.env` file in the project root for more news sources
- **Free Sources**: yfinance and CryptoCompare work without API keys
- **Training**: Model training is optional and can take a long time
- **Predictions**: Requires trained models

For more information, see:
- `QUICK_API_SETUP.md` - API key setup guide
- `ai_stock_predictor_gui.py` - Streamlit GUI version

---